<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/09_validate_hallmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gseapy
from google.colab import drive
drive.mount('/content/drive')

import json, pandas as pd, numpy as np
BASE = '/content/drive/MyDrive/longevity_project'
hallmarks = json.load(open(f'{BASE}/data/processed/hallmarks.json'))
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))
vcap = pd.read_csv(f'{BASE}/data/processed/vcap_sig_info.csv')
genes = pd.read_csv(f'{BASE}/data/raw/GSE92742_Broad_LINCS_gene_info.txt.gz', sep='\t')

bing = genes[genes['pr_is_bing'] == 1]
id_to_symbol = dict(zip(bing['pr_gene_id'].astype(str), bing['pr_gene_symbol']))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.7/689.7 kB 29.0 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
pilot = pd.read_csv(f'{BASE}/data/processed/senescence_pilot_drugs.csv')['name'].tolist()
known_names = {d for k in kd['by_hallmark'].values() for d in k['helpful'] + k['harmful']}
names = set(pilot) | known_names

named = vcap[~vcap['pert_iname'].str.startswith('BRD-')].copy()
named['name'] = named['pert_iname'].str.lower()
pick = named[named['name'].isin(names)].copy()
pick['is_24h'] = pick['pert_time'] == 24
pick = pick.sort_values(['is_24h', 'pert_dose'], ascending=False).groupby('name').head(1)

print(len(pick), 'drugs to score')
print('Missing:', names - set(pick['name']) or 'none')

131 drugs to score
Missing: none


In [3]:
sigs = pd.read_parquet(f'{BASE}/data/processed/vcap_signatures.parquet', columns=list(id_to_symbol))
sigs = sigs.loc[pick['sig_id']]
sigs.columns = [id_to_symbol[c] for c in sigs.columns]
print(sigs.shape)

(131, 10174)


In [4]:
import gseapy as gp
from tqdm import tqdm

gene_sets = {}
for name, hm in hallmarks.items():
    gene_sets[f'{name}|original'] = hm['genes']
    gene_sets[f'{name}|refined'] = hm['genes_refined']

rows = []
for sig_id, row in tqdm(sigs.iterrows(), total=len(sigs)):
    res = gp.prerank(rnk=row.sort_values(ascending=False), gene_sets=gene_sets,
                     permutation_num=100, min_size=5, max_size=1000,
                     seed=0, outdir=None, verbose=False)
    rows.append({'sig_id': sig_id, **res.res2d.set_index('Term')['NES'].astype(float).to_dict()})

nes = pd.DataFrame(rows).set_index('sig_id').join(pick.set_index('sig_id')['name']).set_index('name')
nes.to_csv(f'{BASE}/results/validation_nes.csv')
print(nes.shape)

  0%|          | 0/131 [00:00<?, ?it/s]2026-10-09 18:27:22,911 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  1%|          | 1/131 [00:00<00:21,  6.18it/s]2026-10-09 18:27:23,063 [WARNING] Duplicated values found in preranked stats: 0.01% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  2%|▏         | 2/131 [00:00<00:20,  6.23it/s]2026-10-09 18:27:23,222 [WARNING] Duplicated values found in preranked stats: 0.03% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  2%|▏         | 3/131 [00:00<00:19,  6.53it/s]2026-10-09 18:27:23,367 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  3%|▎         | 4/131 [00:00<00:19,  6.57it/s]2026-10-09 18:27:23,519 [WARNING] Duplicated values found in preranked st

(131, 12)


In [5]:
scores = nes.copy()
for col in scores.columns:
    if hallmarks[col.split('|')[0]]['good'] == 'down':
        scores[col] = -scores[col]
pct = (scores.rank(pct=True) * 100).round(1)
pct.to_csv(f'{BASE}/results/validation_percentiles.csv')

In [6]:
rows = []
for hm, k in kd['by_hallmark'].items():
    for version in ['original', 'refined']:
        col = f'{hm}|{version}'
        helpful = [d for d in k['helpful'] if d in pct.index]
        harmful = [d for d in k['harmful'] if d in pct.index]
        h = pct.loc[helpful, col].mean() if helpful else np.nan
        b = pct.loc[harmful, col].mean() if harmful else np.nan
        rows.append({'hallmark': hm, 'version': version, 'helpful_avg': h, 'harmful_avg': b,
                     'gap': h - (b if harmful else 50)})

summary = pd.DataFrame(rows).round(1)
print(summary.to_string(index=False))

        hallmark  version  helpful_avg  harmful_avg  gap
      senescence original         46.8         35.6 11.2
      senescence  refined         40.2         34.2  5.9
nutrient_sensing original         69.5          NaN 19.5
nutrient_sensing  refined         69.5          NaN 19.5
    mitochondria original         79.8         50.4 29.4
    mitochondria  refined         81.5         50.4 31.2
       autophagy original         52.8          NaN  2.8
       autophagy  refined         52.8          NaN  2.8
    inflammation original         51.6         25.8 25.8
    inflammation  refined         52.7         25.8 26.9
    proteostasis original         64.9         32.4 32.5
    proteostasis  refined         65.4         27.8 37.5


In [7]:
for hm in hallmarks:
    g = summary[summary['hallmark'] == hm].set_index('version')['gap']
    choice = 'refined' if g['refined'] > g['original'] + 5 else 'original'
    hallmarks[hm]['genes_final'] = hallmarks[hm][f'genes{"_refined" if choice == "refined" else ""}']
    hallmarks[hm]['validation'] = {'version': choice, 'gap': float(g[choice])}
    print(f'{hm:17} using {choice:8} gap = {g[choice]:.1f}')

json.dump(hallmarks, open(f'{BASE}/data/processed/hallmarks.json', 'w'), indent=2)

senescence        using original gap = 11.2
nutrient_sensing  using original gap = 19.5
mitochondria      using original gap = 29.4
autophagy         using original gap = 2.8
inflammation      using original gap = 25.8
proteostasis      using original gap = 32.5


In [8]:
for hm, k in kd['by_hallmark'].items():
    col = f"{hm}|{hallmarks[hm]['validation']['version']}"
    wrong = [f'{d} ({pct.loc[d, col]:.0f})' for d in k['helpful'] if d in pct.index and pct.loc[d, col] < 50] + \
            [f'{d} ({pct.loc[d, col]:.0f})' for d in k['harmful'] if d in pct.index and pct.loc[d, col] > 50]
    print(f"{hm:17} on the wrong side: {', '.join(wrong) or 'none'}")

senescence        on the wrong side: curcumin (29), apigenin (48), ruxolitinib (24), simvastatin (49), temozolomide (62)
nutrient_sensing  on the wrong side: everolimus (49), resveratrol (6), curcumin (47)
mitochondria      on the wrong side: rotenone (56)
autophagy         on the wrong side: resveratrol (46), metformin (3), carbamazepine (6)
inflammation      on the wrong side: curcumin (5), quercetin (40), metformin (37), parthenolide (14), simvastatin (25)
proteostasis      on the wrong side: metformin (40), thapsigargin (67), tanespimycin (76), chloroquine (61)


In [9]:
from google.colab import drive
drive.mount('/content/drive')
import json
BASE = '/content/drive/MyDrive/longevity_project'
h = json.load(open(f'{BASE}/data/processed/hallmarks.json'))

gaps = ', '.join(f"{name.replace('_', ' ')} {v['validation']['gap']:.0f}" for name, v in h.items())
refined = [name.replace('_', ' ') for name, v in h.items() if v['validation']['version'] == 'refined']
print(f"Validated all six hallmarks against 37 known drugs. Gaps: {gaps}. "
      f"Using refined gene lists for {', '.join(refined) if refined else 'none (original lists kept for all six)'}.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Validated all six hallmarks against 37 known drugs. Gaps: senescence 11, nutrient sensing 20, mitochondria 29, autophagy 3, inflammation 26, proteostasis 32. Using refined gene lists for none (original lists kept for all six).
